In [37]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers
from tqdm import tqdm
import time

es = Elasticsearch("http://localhost:9200")
index_name = "recipes"

index_config = {
    "settings": {
        "analysis": {
            "filter": {
                "autocomplete_filter": {
                    "type": "edge_ngram",
                    "min_gram": 2,
                    "max_gram": 20
                },
                "word_splitter": {
                    "type": "word_delimiter_graph",
                    "preserve_original": True,
                    "catenate_words": True
                }
            },
            "analyzer": {
                "compound_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "word_splitter"]
                },
                "autocomplete_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "autocomplete_filter"]
                },
                "search_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "RecipeId": {"type": "integer"},

            "Name": {
                "type": "text",
                "analyzer": "compound_analyzer",
                "fields": {
                    "autocomplete": {
                        "type": "text",
                        "analyzer": "autocomplete_analyzer",
                        "search_analyzer": "search_analyzer"
                    },
                    "english": {
                        "type": "text",
                        "analyzer": "english"
                    }
                }
            },

            "NameSuggest": {
                "type": "completion"
            },

            "RecipeIngredientParts": {
                "type": "text",
                "analyzer": "english",
                "term_vector": "yes"
            },

            "Description": {"type": "text", "analyzer": "english"},
            "RecipeInstructions": {"type": "text", "analyzer": "english"},
            "Keywords": {"type": "text", "analyzer": "english"},
            "RecipeCategory": {"type": "keyword"},
            "Images": {"type": "keyword", "index": False},
            "AggregatedRating": {"type": "float"},
            "Calories": {"type": "float"},
            "FatContent": {"type": "float"},
            "SaturatedFatContent": {"type": "float"},
            "CholesterolContent": {"type": "float"},
            "SodiumContent": {"type": "float"},
            "CarbohydrateContent": {"type": "float"},
            "FiberContent": {"type": "float"},
            "SugarContent": {"type": "float"},
            "ProteinContent": {"type": "float"},
            "CookTimeMins": {"type": "integer"},
            "PrepTimeMins": {"type": "integer"},
            "TotalTimeMins": {"type": "integer"}
        }
    }
}

print("Resetting index...")
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

es.indices.create(
    index=index_name,
    settings=index_config["settings"],
    mappings=index_config["mappings"]
)

print("Loading data...")
df = pd.read_csv('../data/recipes_ready_for_es.csv')
df = df.drop_duplicates(subset=["RecipeId"])
print(f"Total unique recipes: {len(df)}")

def safe_float(v):
    try:
        return float(v) if pd.notna(v) and v != "" else 0.0
    except:
        return 0.0

def safe_int(v):
    try:
        return int(float(v)) if pd.notna(v) and v != "" else 0
    except:
        return 0

float_fields = [
    'AggregatedRating', 'Calories', 'FatContent', 'SaturatedFatContent',
    'CholesterolContent', 'SodiumContent', 'CarbohydrateContent',
    'FiberContent', 'SugarContent', 'ProteinContent'
]

int_fields = ['CookTimeMins', 'PrepTimeMins', 'TotalTimeMins', 'RecipeId']

def generate_docs(dataframe):
    for row in dataframe.to_dict(orient="records"):
        doc = row
        for f in float_fields:
            doc[f] = safe_float(doc.get(f))
        for f in int_fields:
            doc[f] = safe_int(doc.get(f))

        for k, v in doc.items():
            if pd.isna(v):
                doc[k] = ""

        if isinstance(doc.get('Images'), str) and len(doc['Images']) > 10000:
            doc['Images'] = doc['Images'][:10000]

        if isinstance(doc.get('RecipeCategory'), str) and len(doc['RecipeCategory']) > 10000:
            doc['RecipeCategory'] = doc['RecipeCategory'][:10000]

        if doc.get("Name") and isinstance(doc.get("Name"), str):
            doc["NameSuggest"] = {"input": doc["Name"]}

        yield {
            "_index": index_name,
            "_id": doc["RecipeId"],
            "_source": doc
        }

print(f"Indexing {len(df)} documents...")
start_time = time.time()

success = 0
failed = 0

for ok, result in tqdm(
    helpers.streaming_bulk(
        es,
        generate_docs(df),
        chunk_size=1000,
        request_timeout=120
    ),
    total=len(df)
):
    if ok:
        success += 1
    else:
        failed += 1

end_time = time.time()

print(f"\nCompleted in {end_time - start_time:.2f} seconds")
print(f"Success: {success}")
print(f"Failed: {failed}")

count = es.count(index=index_name)["count"]
print(f"Documents in Elasticsearch: {count}")

Resetting index...
Loading data...
Total unique recipes: 522517
Indexing 522517 documents...


100%|██████████| 522517/522517 [02:34<00:00, 3379.77it/s]



Completed in 154.62 seconds
Success: 522517
Failed: 0
Documents in Elasticsearch: 522517


In [64]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")
index_name = "recipes"

search_keyword = "Ryugyu"

print(f"Searching for recipes related to: '{search_keyword}'\n")

search_query = {
    "query": {
        "bool": {
            "should": [
                {
                    "multi_match": {
                        "query": search_keyword,
                        "fields": ["Name^3", "RecipeIngredientParts^2", "RecipeInstructions"],
                        "fuzziness": "AUTO",
                        "operator": "or"
                    }
                },
                {
                    "match": {
                        "Name.autocomplete": {
                            "query": search_keyword,
                            "boost": 2
                        }
                    }
                }
            ],
            "minimum_should_match": 1
        }
    },
    "size": 5
}

response = es.search(index=index_name, body=search_query)

hits = response["hits"]["hits"]
total_matches = response["hits"]["total"]["value"]

print(f"Total matches: {total_matches}")
print("=" * 60)

for idx, hit in enumerate(hits, start=1):
    score = hit["_score"]
    source = hit["_source"]

    print(f"Rank {idx} | Score: {score:.4f}")
    print(f"Recipe name: {source.get('Name', 'N/A')}")
    print(f"Total time: {source.get('TotalTimeMins', 'N/A')} minutes")
    print(f"Category: {source.get('RecipeCategory', 'N/A')}")

    ingredients = source.get("RecipeIngredientParts", "")
    short_ingredients = ingredients[:100] + "..." if len(ingredients) > 100 else ingredients
    print(f"Ingredients: {short_ingredients}")
    print("-" * 60)

Searching for recipes related to: 'Ryugyu'

Total matches: 2
Rank 1 | Score: 19.4896
Recipe name: Venison Rugu  Ragu  Ragout
Total time: 150 minutes
Category: One Dish Meal
Ingredients: ground beef, white mushrooms, onion, canola oil, butter, salt, pepper
------------------------------------------------------------
Rank 2 | Score: 6.6016
Recipe name: Ceviche De Pescado (Fish Salad Cooked in Lime Juice)
Total time: 125 minutes
Category: Yam/Sweet Potato
Ingredients: salt, fresh lime juice, salt, garlic clove, fresh chili pepper, parsley, cilantro, onion, lettuce le...
------------------------------------------------------------
